# Pilot Tests & Exploration

**Sections:**
1. Model inference — loading and running a model with Graphium
2. Fine-tuning on TDC ADMET — data module demo (WIP)
3. Pre-training embedding exploration — MACE-OFF and BBBC047/CPCNN

---
## 1. Model Inference

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from os.path import dirname, abspath
from copy import deepcopy

import omegaconf
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir

import graphium
from graphium.config._loader import (
    load_accelerator,
    load_datamodule,
    load_architecture,
    load_metrics,
    load_predictor,
)

### Read the config file

In [ ]:
# Set up the working directory to graphium project root
MAIN_DIR = dirname(dirname(abspath(graphium.__file__)))
os.chdir(MAIN_DIR)

# Load the Hydra config (toymix + GCN as a minimal example)
config_dir = os.path.join(MAIN_DIR, "expts", "hydra-configs")
with initialize_config_dir(version_base=None, config_dir=config_dir):
    cfg = compose(config_name="main", overrides=["model=gcn", "accelerator=cpu"])

cfg = OmegaConf.to_container(cfg, resolve=True)
print("Config loaded. Keys:", list(cfg.keys()))

### Load a dataset

In [ ]:
# Load accelerator config and create the datamodule
cfg, accelerator_type = load_accelerator(cfg)

datamodule = load_datamodule(cfg, accelerator_type)
datamodule.prepare_data()

print(f"Accelerator: {accelerator_type}")
print(f"Datamodule type: {type(datamodule).__name__}")
print(f"Input dims: {datamodule.in_dims}")

In [ ]:
# Build the model architecture from config
model_class, model_kwargs = load_architecture(cfg, in_dims=datamodule.in_dims)

model = model_class(**model_kwargs)
print(f"\nModel class: {model_class.__name__}")
print(model)

In [ ]:
# Load metrics
metrics = load_metrics(cfg)
for task, task_metrics in metrics.items():
    print(f"{task}: {list(task_metrics.keys())}")

In [ ]:
# Build the full predictor (model + optimizer + metrics)
predictor = load_predictor(
    config=cfg,
    model_class=model_class,
    model_kwargs=model_kwargs,
    metrics=metrics,
    task_levels=datamodule.get_task_levels(),
    accelerator_type=accelerator_type,
    featurization=datamodule.featurization,
    task_norms=datamodule.task_norms,
)

print(f"Predictor ready. Tasks: {list(predictor.model.task_heads.keys())}")

---
## 2. Fine-tuning on TDC ADMET Benchmarks (WIP)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml
import omegaconf
from datetime import datetime

from typing import Union, List
from copy import deepcopy
from tdc.utils import retrieve_benchmark_names

from graphium.config._loader import (
    load_datamodule,
    load_metrics,
    load_architecture,
    load_predictor,
    load_trainer,
    save_params_to_wandb,
    load_accelerator,
    load_yaml_config,
)

In [ ]:
# First, let's read the yaml configuration file
with open("../expts/configs/config_tdc_admet_demo.yaml", "r") as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

### Get all TDC benchmark names

In [ ]:
benchmarks = retrieve_benchmark_names("admet_group")
len(benchmarks)

While there is a total of 22, let's just use two for practicality sake: One regression and one classification task! 

In [ ]:
benchmarks = ["caco2_wang", "hia_hou"]

### Initialize all training components per task
**NOTE**: Since we do not have fine-tuning logic, this for now just creates a new model. Ultimately, we will want to use fine-tuning code to evaluate how well the pre-trained model transfers to downstream tasks. 

In [ ]:
def training_testing_loop(cfg):
    """
    Simple loop to train a model from scratch and test it. 
    """
    
    # Initialize object from config
    cfg, accelerator_type = load_accelerator(cfg)
    datamodule = load_datamodule(cfg, accelerator_type)
    model_class, model_kwargs = load_architecture(cfg, in_dims=datamodule.in_dims)
    metrics = load_metrics(cfg)
    
    # Prepare data
    datamodule.prepare_data()
    
    # Initialize the predictor
    predictor = load_predictor(
        cfg,
        model_class,
        model_kwargs,
        metrics,
        datamodule.get_task_levels(),
        accelerator_type,
        datamodule.featurization,
        datamodule.task_norms
    )
    
    # Initialize the trainer
    date_time_suffix = datetime.now().strftime("%d.%m.%Y_%H.%M.%S")
    trainer = load_trainer(cfg, "tdc-admet", accelerator_type, date_time_suffix)
        
    # Train
    predictor.set_max_nodes_edges_per_graph(datamodule, stages=["train", "val"])
    trainer.fit(model=predictor, datamodule=datamodule)
    
    # Test
    predictor.set_max_nodes_edges_per_graph(datamodule, stages=["test"])
    results = trainer.test(model=predictor, datamodule=datamodule)
    
    return results


In [ ]:
def filter_cfg_based_on_benchmark_name(config, names: Union[List[str], str]):
    """
    Filter a base config for the full TDC ADMET benchmarking group to only 
    have settings related to a subset of the endpoints
    """
    
    if config["datamodule"]["module_type"] != "ADMETBenchmarkDataModule":
        raise ValueError("You can only use this method for the `ADMETBenchmarkDataModule`")
        
    if isinstance(names, str):
        names = [names]
    
    def _filter(d):
        return {k: v for k, v in d.items() if k in names}
         
    cfg = deepcopy(config)
    
    # Update the datamodule arguments
    cfg["datamodule"]["args"]["tdc_benchmark_names"] = names
    
    # Filter the relevant config sections
    cfg["architecture"]["task_heads"] = _filter(cfg["architecture"]["task_heads"])
    cfg["predictor"]["metrics_on_progress_bar"] = _filter(cfg["predictor"]["metrics_on_progress_bar"])
    cfg["predictor"]["loss_fun"] = _filter(cfg["predictor"]["loss_fun"])
    cfg["metrics"] = _filter(cfg["metrics"])
    
    return cfg

In [ ]:
results = {}

for name in benchmarks: 
    
    # Run the training-testing loop
    cfg = filter_cfg_based_on_benchmark_name(config, name)
    benchmark_results = training_testing_loop(cfg)
    
    # Extract the main metric from the config
    metric = cfg["predictor"]["metrics_on_progress_bar"][name][0]
    key = f"graph_{name}/{metric}/test"
    results[f"{name}/{metric}"] = benchmark_results[0][key]

In [ ]:
print(omegaconf.OmegaConf.to_yaml(results))

The End. 

---
## 3. Pre-training Embedding Exploration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Project paths
PROJECT_ROOT = Path("/home/shpark/prj-molrepr")
GRAPHIUM_DIR = PROJECT_ROOT / "graphium"
DATA_DIR = PROJECT_ROOT / "data"

sys.path.insert(0, str(GRAPHIUM_DIR))

---
### Part 1: MLIP Embeddings (MACE-OFF)

**Motivation**: MLIPs encode 3D conformational and quantum-chemical information learned from DFT calculations. By distilling MACE-OFF embeddings into a 2D GNN via pre-training, we inject 3D-aware knowledge without requiring 3D coordinates at inference time.

**Pipeline**: SMILES → RDKit 3D conformer → MACE-OFF → pool atom embeddings → molecular vector

**MACE-OFF23**: Trained on organic molecules (SPICE dataset) at coupled-cluster accuracy. Produces 128-dim per-atom embeddings.

#### 1.1 Install and load MACE-OFF

In [ ]:
# Install MACE if not present
# !pip install mace-torch

import torch
from rdkit import Chem
from rdkit.Chem import AllChem
import ase
from ase import Atoms

try:
    from mace.calculators import mace_off
    print("MACE-OFF loaded successfully")
except ImportError:
    print("Install mace-torch: pip install mace-torch")
    raise

#### 1.2 SMILES → 3D conformer → MACE embedding pipeline

In [ ]:
def smiles_to_ase_atoms(smiles: str, n_conformers: int = 1, seed: int = 42) -> list:
    """Convert SMILES to ASE Atoms via RDKit 3D embedding."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    mol = Chem.AddHs(mol)
    
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.numThreads = 1
    
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, params=params)
    if len(conf_ids) == 0:
        return []
    
    # Optimize with MMFF
    for cid in conf_ids:
        try:
            AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=200)
        except Exception:
            pass
    
    atoms_list = []
    for cid in conf_ids:
        conf = mol.GetConformer(cid)
        positions = conf.GetPositions()
        symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
        atoms_list.append(Atoms(symbols=symbols, positions=positions))
    
    return atoms_list


# Test with a simple molecule
test_smiles = "c1ccccc1"  # benzene
atoms_list = smiles_to_ase_atoms(test_smiles)
print(f"Benzene: {len(atoms_list)} conformer(s), {len(atoms_list[0])} atoms")
print(f"Symbols: {atoms_list[0].get_chemical_symbols()}")
print(f"Positions shape: {atoms_list[0].positions.shape}")

In [ ]:
def extract_mace_embedding(calc, atoms, pool="sum", heavy_atoms_only=True):
    """Extract molecular embedding from MACE-OFF model.
    
    Uses calc.get_descriptors() which returns per-atom node features,
    then pools to a molecular vector.
    
    Args:
        calc: MACECalculator instance
        atoms: ASE Atoms object (with H atoms)
        pool: "mean" or "sum" pooling. Sum is preferred — it matches
            MACE's physics (energy = sum of atomic contributions) and
            encodes molecule size, giving richer pre-training signal.
        heavy_atoms_only: if True, exclude H atoms from pooling.
            H atoms dominate organic molecules and have very similar
            local environments, washing out the structural signal.
    
    Returns:
        np.ndarray of shape (embedding_dim,) — pooled molecular embedding
    """
    # get_descriptors returns (n_atoms, embedding_dim)
    node_feats = calc.get_descriptors(atoms)
    
    if heavy_atoms_only:
        heavy_mask = np.array([s != "H" for s in atoms.get_chemical_symbols()])
        node_feats = node_feats[heavy_mask]
    
    if pool == "mean":
        return node_feats.mean(axis=0)
    elif pool == "sum":
        return node_feats.sum(axis=0)
    else:
        raise ValueError(f"Unknown pooling: {pool}")


# Initialize MACE-OFF calculator — use the large model (4.7M params, 448-dim embeddings)
# small=192d, medium=256d, large=448d
device = "cuda" if torch.cuda.is_available() else "cpu"
calc = mace_off(model="large", device=device, default_dtype="float64")
print(f"MACE-OFF large loaded (device={device})")

# Quick check: embedding dim
from ase import Atoms as _Atoms
_test = _Atoms("H2", positions=[[0,0,0],[0,0,0.74]])
_desc = calc.get_descriptors(_test)
print(f"Embedding dim per atom: {_desc.shape[1]}")
print(f"Pooling: sum over heavy atoms (encodes molecule size + local chemistry)")

#### 1.3 Test on sample molecules

In [ ]:
# Load ADMET data from TDC and sample 10 molecules
from tdc.benchmark_group import admet_group

group = admet_group(path="/tmp/tdc_admet_cache")
benchmark = group.get("caco2_wang")
train_df, val_df = group.get_train_valid_split(seed=42, benchmark="caco2_wang")

# Sample 10 random molecules from training set
sample_df = train_df.sample(n=10, random_state=42).reset_index(drop=True)
print(f"Sampled {len(sample_df)} molecules from caco2_wang training set ({len(train_df)} total)")
print(sample_df[["Drug_ID", "Drug", "Y"]].to_string())

In [ ]:
# Extract MACE embeddings for the 10 sampled ADMET molecules
admet_embeddings = {}
for idx, row in sample_df.iterrows():
    smi = row["Drug"]
    name = row["Drug_ID"]
    atoms_list = smiles_to_ase_atoms(smi)
    if atoms_list:
        emb = extract_mace_embedding(calc, atoms_list[0])
        admet_embeddings[name] = emb
        print(f"{idx:2d} | {name:20s} | atoms={len(atoms_list[0]):3d} | Y={row['Y']:.2f} | emb_norm={np.linalg.norm(emb):.3f}")
    else:
        print(f"{idx:2d} | {name:20s} | FAILED")

print(f"\nSuccessfully embedded {len(admet_embeddings)}/{len(sample_df)} molecules")

#### 1.3c Visualize 3D conformations used for embeddings

Verify that RDKit's ETKDGv3 is generating reasonable 3D conformers. ETKDGv3 uses experimental torsional-angle preferences from the Cambridge Structural Database (CSD), producing more physically realistic conformations than earlier methods.

In [ ]:
from rdkit.Chem import Draw, AllChem
import py3Dmol

def generate_conformers(smiles, n_conformers=5, seed=42):
    """Generate multiple 3D conformers using ETKDGv3 (CSD torsional preferences)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, []
    mol = Chem.AddHs(mol)
    
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.numThreads = 1
    params.pruneRmsThresh = 0.5  # prune conformers within 0.5 Å RMSD
    
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, params=params)
    
    # Optimize each conformer with MMFF94
    energies = []
    for cid in conf_ids:
        AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=500)
        ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid)
        energies.append((cid, ff.CalcEnergy() if ff else float('inf')))
    energies.sort(key=lambda x: x[1])
    
    return mol, energies


# Generate conformers for all 10 ADMET molecules
N_CONF = 5
all_mols = {}  # name -> (mol, energies, smiles)

for idx, row in sample_df.iterrows():
    smi = row["Drug"]
    name = row["Drug_ID"]
    mol, energies = generate_conformers(smi, n_conformers=N_CONF)
    if mol is not None:
        all_mols[name] = (mol, energies, smi)
        print(f"{name:35s} | {len(energies)} conformers | E range: {energies[0][1]:.1f} — {energies[-1][1]:.1f} kcal/mol")

In [ ]:
# Visualize all conformers for each ADMET molecule (interactive 3D)
def show_conformers_grid(mol, energies, name):
    """Show all conformers side by side in a single py3Dmol viewer."""
    n = len(energies)
    viewer = py3Dmol.view(width=900, height=300, viewergrid=(1, min(n, 5)))
    for i, (cid, e) in enumerate(energies[:5]):
        mb = Chem.MolToMolBlock(mol, confId=cid)
        viewer.addModel(mb, "mol", viewer=(0, i))
        viewer.setStyle({"stick": {"colorscheme": "Jmol"}}, viewer=(0, i))
        viewer.addLabel(f"E={e:.0f}", {"fontSize": 10, "backgroundColor": "white"}, viewer=(0, i))
        viewer.zoomTo(viewer=(0, i))
    return viewer

for name, (mol, energies, smi) in all_mols.items():
    print(f"\n{name} — {len(energies)} conformers")
    viewer = show_conformers_grid(mol, energies, name)
    viewer.show()

In [ ]:
# Extract MACE embeddings for ALL conformers of ALL 10 molecules
from scipy.spatial.distance import cosine

all_conf_embeddings = {}  # name -> list of (conf_id, energy, embedding)

for name, (mol, energies, smi) in all_mols.items():
    conf_embs = []
    for cid, e in energies:
        conf = mol.GetConformer(cid)
        atoms = Atoms(
            symbols=[a.GetSymbol() for a in mol.GetAtoms()],
            positions=conf.GetPositions()
        )
        emb = extract_mace_embedding(calc, atoms)
        conf_embs.append((cid, e, emb))
    all_conf_embeddings[name] = conf_embs
    print(f"{name:35s} | {len(conf_embs)} embeddings extracted")

print(f"\nTotal: {sum(len(v) for v in all_conf_embeddings.values())} embeddings across {len(all_conf_embeddings)} molecules")

In [ ]:
# Intra-molecule similarity (same molecule, different conformers)
print("=== Intra-molecule: cosine similarity between conformers ===\n")

intra_sims = {}
for name, conf_embs in all_conf_embeddings.items():
    embs = [e for _, _, e in conf_embs]
    n = len(embs)
    if n < 2:
        print(f"{name:35s} | only 1 conformer")
        continue
    sims = []
    for i in range(n):
        for j in range(i+1, n):
            sims.append(1 - cosine(embs[i], embs[j]))
    intra_sims[name] = sims
    print(f"{name:35s} | n_conf={n} | sim: mean={np.mean(sims):.6f}  min={np.min(sims):.6f}  max={np.max(sims):.6f}")

print(f"\nOverall intra-molecule similarity: {np.mean([s for sims in intra_sims.values() for s in sims]):.6f}")

In [ ]:
# Inter-molecule similarity (different molecules, lowest-energy conformer)
names = list(all_conf_embeddings.keys())
n_mol = len(names)

# Use lowest-energy conformer for each molecule
best_embs = {name: conf_embs[0][2] for name, conf_embs in all_conf_embeddings.items()}

inter_sim = np.zeros((n_mol, n_mol))
for i in range(n_mol):
    for j in range(n_mol):
        inter_sim[i, j] = 1 - cosine(best_embs[names[i]], best_embs[names[j]])

print("=== Inter-molecule: cosine similarity (lowest-energy conformer) ===\n")

# Print as table
off_diag = inter_sim[np.triu_indices(n_mol, k=1)]
print(f"Off-diagonal stats: mean={off_diag.mean():.4f}  min={off_diag.min():.4f}  max={off_diag.max():.4f}  std={off_diag.std():.4f}")

# Compare with intra-molecule stats
all_intra = [s for sims in intra_sims.values() for s in sims]
print(f"Intra-molecule stats: mean={np.mean(all_intra):.4f}  min={np.min(all_intra):.4f}")
print(f"\nSeparation: intra_min ({np.min(all_intra):.4f}) vs inter_max ({off_diag.max():.4f})")

In [ ]:
# Full heatmap: ALL conformers of ALL molecules
# Each row/column = one conformer of one molecule
# Block-diagonal = intra-molecule, off-diagonal = inter-molecule

import matplotlib.pyplot as plt

all_labels = []
all_emb_list = []
mol_boundaries = []  # for drawing separator lines

for name in names:
    mol_boundaries.append(len(all_labels))
    for cid, e, emb in all_conf_embeddings[name]:
        short_name = name[:12] if len(name) > 12 else name
        all_labels.append(f"{short_name}_c{cid}")
        all_emb_list.append(emb)

all_emb_arr = np.stack(all_emb_list)
N = len(all_emb_arr)

# Compute full similarity matrix
full_sim = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        full_sim[i, j] = 1 - cosine(all_emb_arr[i], all_emb_arr[j])

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(full_sim, cmap="RdBu_r", vmin=0.7, vmax=1.0)

# Draw molecule boundary lines
for b in mol_boundaries[1:]:
    ax.axhline(b - 0.5, color="black", linewidth=1.5)
    ax.axvline(b - 0.5, color="black", linewidth=1.5)

# Label with molecule names at block centers
mol_centers = []
for i in range(len(mol_boundaries)):
    start = mol_boundaries[i]
    end = mol_boundaries[i+1] if i+1 < len(mol_boundaries) else N
    mol_centers.append((start + end) / 2)

ax.set_xticks(mol_centers)
ax.set_xticklabels([n[:15] for n in names], rotation=45, ha="right", fontsize=8)
ax.set_yticks(mol_centers)
ax.set_yticklabels([n[:15] for n in names], fontsize=8)

plt.colorbar(im, label="Cosine similarity", shrink=0.8)
ax.set_title("MACE-OFF Embedding Similarity: All Conformers of 10 ADMET Molecules\n"
             "(block diagonal = same molecule, off-diagonal = different molecules)", fontsize=11)
plt.tight_layout()
plt.show()

# Summary
print(f"Matrix size: {N}x{N} ({N} total conformers from {n_mol} molecules)")
print(f"Block diagonal (intra-molecule):  mean={np.mean(all_intra):.4f}")
print(f"Off-block (inter-molecule):       mean={off_diag.mean():.4f}")

In [ ]:
# Visualize cosine similarity matrix for ADMET molecules
from scipy.spatial.distance import cosine

names = list(admet_embeddings.keys())
n = len(names)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = 1 - cosine(admet_embeddings[names[i]], admet_embeddings[names[j]])

# Also get the Y values for annotation
y_vals = {row["Drug_ID"]: row["Y"] for _, row in sample_df.iterrows()}
labels = [f"{nm}\n(Y={y_vals.get(nm, 0):.1f})" for nm in names]

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="Cosine similarity", shrink=0.8)
plt.title("MACE-OFF Embedding Similarity — ADMET (caco2_wang) Training Molecules")
plt.tight_layout()
plt.show()

# Print correlation between embedding similarity and Y-value similarity
print("\nEmbedding similarity vs. property similarity:")
y_arr = np.array([y_vals[nm] for nm in names])
y_diff = np.abs(y_arr[:, None] - y_arr[None, :])
mask = np.triu_indices(n, k=1)
from scipy.stats import pearsonr, spearmanr
r_p, p_p = pearsonr(sim_matrix[mask], -y_diff[mask])  # negative because similar Y → small diff
r_s, p_s = spearmanr(sim_matrix[mask], -y_diff[mask])
print(f"  Pearson r={r_p:.3f} (p={p_p:.3f})")
print(f"  Spearman r={r_s:.3f} (p={p_s:.3f})")
print("  (Positive correlation means similar MACE embeddings → similar caco2 permeability)")

#### 1.4 Batch extraction for a dataset

Generate MACE-OFF embeddings for all molecules in a dataset (e.g., the ADMET molecules or LargeMix SMILES). This can be used as a pre-training target alongside RxRx3 and DTI.

In [ ]:
def batch_extract_mace_embeddings(smiles_list, calc, pool="mean", n_conformers=1):
    """Extract MACE-OFF embeddings for a list of SMILES.
    
    Returns:
        embeddings: np.ndarray of shape (n_molecules, embedding_dim)
        valid_idx: list of indices that succeeded
        failed_idx: list of indices that failed
    """
    all_embeddings = []
    valid_idx = []
    failed_idx = []
    
    for i, smi in enumerate(tqdm(smiles_list, desc="Extracting MACE embeddings")):
        try:
            atoms_list = smiles_to_ase_atoms(smi, n_conformers=n_conformers)
            if not atoms_list:
                failed_idx.append(i)
                continue
            
            if n_conformers == 1:
                emb = extract_mace_embedding(calc, atoms_list[0], pool=pool)
            else:
                # Average over conformers
                emb = np.mean([
                    extract_mace_embedding(calc, atoms, pool=pool)
                    for atoms in atoms_list
                ], axis=0)
            
            all_embeddings.append(emb)
            valid_idx.append(i)
        except Exception as e:
            failed_idx.append(i)
            if i < 5:  # Print first few errors
                print(f"  [{i}] Failed: {smi[:50]}... | {e}")
    
    if all_embeddings:
        embeddings = np.stack(all_embeddings)
    else:
        embeddings = np.array([])
    
    print(f"\nSuccess: {len(valid_idx)}/{len(smiles_list)} | Failed: {len(failed_idx)}")
    return embeddings, valid_idx, failed_idx


# Example: extract for a small sample from QM9
# qm9_df = pd.read_csv(DATA_DIR / "graphium/neurips2023/small-dataset/qm9.csv", nrows=100)
# qm9_embeddings, valid, failed = batch_extract_mace_embeddings(qm9_df["smiles"].tolist(), calc)
# print(f"Embedding shape: {qm9_embeddings.shape}")  # Expected: (n_valid, 192)

#### 1.5 Save embeddings in Graphium-compatible format

To use as a pre-training target, save as a CSV with SMILES + embedding columns (same format as RxRx3).

In [ ]:
def save_embeddings_csv(smiles_list, embeddings, valid_idx, output_path, prefix="mace"):
    """Save embeddings as CSV compatible with Graphium's task_specific_args format.
    
    Output format matches RxRx3: SMILES column + feature_0, feature_1, ... columns.
    """
    valid_smiles = [smiles_list[i] for i in valid_idx]
    emb_dim = embeddings.shape[1]
    
    col_names = [f"{prefix}_{i}" for i in range(emb_dim)]
    df = pd.DataFrame(embeddings, columns=col_names)
    df.insert(0, "SMILES", valid_smiles)
    
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df)} molecules x {emb_dim} dims to {output_path}")
    print(f"Columns: SMILES, {col_names[0]}, ..., {col_names[-1]}")
    return df


# Example usage (uncomment to run):
# output_path = DATA_DIR / "mace_off" / "qm9_mace_embeddings.csv"
# output_path.parent.mkdir(parents=True, exist_ok=True)
# save_embeddings_csv(qm9_df["smiles"].tolist(), qm9_embeddings, valid, output_path)

---
### Part 2: U2OS Cell Morphology — BBBC047

**Motivation**: RxRx3 uses HUVEC cells (~17K compounds, 384-dim). BBBC047 uses U2OS cells (human osteosarcoma) which respond differently to chemical perturbations, providing complementary biological signal. The dataset has substantially more data points.

**BBBC047**: Part of the Broad Bioimage Benchmark Collection. U2OS cells with Cell Painting assay (5-6 fluorescence channels: DAPI, ER, RNA, AGP, Mito).

**Embedding extraction options**:
1. **CellProfiler features** — Classical morphological features (~1000+ dims), may be pre-computed
2. **Deep learning embeddings** — Extract from a pretrained vision model (e.g., MAE, DINO on cell images)
3. **Pre-computed embeddings** — Check if the JUMP Cell Painting consortium provides them

#### Data sources
- BBBC: https://bbbc.broadinstitute.org/BBBC047
- JUMP Cell Painting Gallery (AWS): `s3://cellpainting-gallery/`
- Related: BBBC022 (U2OS, 1571 compounds), cpg0016 (JUMP-CP, ~116K compounds)

#### 2.1 Download JUMP compound metadata and pre-computed CPCNN embeddings

The JUMP Cell Painting consortium (cpg0016) provides **pre-computed 672-dim Cell Painting CNN embeddings** (EfficientNet-B0, Moshkov et al. Nature Comms 2024) on AWS S3. No raw images or DeepProfiler needed.

**Data sources:**
- Compound metadata (SMILES): `github.com/jump-cellpainting/datasets/`
- CPCNN embeddings: `s3://cellpainting-gallery/cpg0016-jump/.../workspace_dl/profiles/cpcnn_zenodo_7114558/`
- 115,796 compounds with SMILES across all sources
- 672-dim per-well embeddings stored as parquet

In [ ]:
# Download JUMP compound metadata
import subprocess, io

JUMP_META_DIR = DATA_DIR / "jump_cellpainting"
JUMP_META_DIR.mkdir(parents=True, exist_ok=True)

# 1. Compound metadata (SMILES, InChIKey, JCP2022 ID)
compound_url = "https://github.com/jump-cellpainting/datasets/raw/main/metadata/compound.csv.gz"
compound_path = JUMP_META_DIR / "compound.csv.gz"
if not compound_path.exists():
    print("Downloading compound metadata...")
    subprocess.run(["wget", "-q", "-O", str(compound_path), compound_url], check=True)
compound_df = pd.read_csv(compound_path)
print(f"Compounds: {len(compound_df)} rows")
print(f"Columns: {list(compound_df.columns)}")
print(f"Sample:\n{compound_df.head(3)}")

# 2. Well metadata (plate/well → compound mapping)
well_url = "https://github.com/jump-cellpainting/datasets/raw/main/metadata/well.csv.gz"
well_path = JUMP_META_DIR / "well.csv.gz"
if not well_path.exists():
    print("\nDownloading well metadata...")
    subprocess.run(["wget", "-q", "-O", str(well_path), well_url], check=True)
well_df = pd.read_csv(well_path)
print(f"\nWells: {len(well_df)} rows")
print(f"Columns: {list(well_df.columns)}")
print(f"Sources: {well_df['Metadata_Source'].nunique()}")
print(f"Compound wells: {(well_df['Metadata_JCP2022'].str.startswith('JCP2022_', na=False)).sum()}")

#### 2.2 Download pilot CPCNN embeddings (5 plates from source_4)

Each plate parquet has 384 wells with 672-dim CPCNN embeddings. Download a small pilot set (~8 MB).

In [ ]:
# List available CPCNN plates from source_4 (Broad)
CPCNN_S3_BASE = "s3://cellpainting-gallery/cpg0016-jump/source_4/workspace_dl/profiles/cpcnn_zenodo_7114558/"
CPCNN_LOCAL = JUMP_META_DIR / "cpcnn_profiles"
CPCNN_LOCAL.mkdir(parents=True, exist_ok=True)

# List batches
result = subprocess.run(
    ["aws", "s3", "ls", CPCNN_S3_BASE, "--no-sign-request"],
    capture_output=True, text=True, timeout=30
)
if result.returncode == 0:
    batches = [line.strip().split()[-1].rstrip("/") for line in result.stdout.strip().split("\n") if "PRE" in line]
    print(f"Found {len(batches)} batches in source_4")
    print(f"First 5: {batches[:5]}")
else:
    print(f"Error listing S3: {result.stderr[:200]}")
    batches = []

#### 2.3 Download and inspect a few plates

In [ ]:
# Download compound-rich plates (plates where wells have SMILES in compound metadata)
N_PILOT_PLATES = 5

# These plates have 384 compound wells each, verified to have SMILES
PILOT_PLATES = [
    ("2021_08_30_Batch13", "BR00127149"),
    ("2021_08_23_Batch12", "BR00126115"),
    ("2021_08_23_Batch12", "BR00126116"),
    ("2021_08_23_Batch12", "BR00126117"),
    ("2021_06_14_Batch6",  "BR00121429"),
]

downloaded = []
for batch_name, plate_name in PILOT_PLATES[:N_PILOT_PLATES]:
    plate_s3 = f"{CPCNN_S3_BASE}{batch_name}/{plate_name}/{plate_name}.parquet"
    plate_local = CPCNN_LOCAL / f"{plate_name}.parquet"
    if not plate_local.exists():
        print(f"  Downloading {plate_name} from {batch_name}...")
        subprocess.run(
            ["aws", "s3", "cp", plate_s3, str(plate_local), "--no-sign-request"],
            capture_output=True, timeout=60
        )
    if plate_local.exists():
        downloaded.append(plate_local)

print(f"Downloaded {len(downloaded)} compound-rich plates")

# Inspect first plate
if downloaded:
    df_plate = pd.read_parquet(downloaded[0])
    print(f"\nPlate columns: {list(df_plate.columns)}")
    print(f"Rows: {len(df_plate)}")
    sample_emb = np.array(df_plate["all_emb"].iloc[0])
    print(f"Embedding dim: {len(sample_emb)}")
    print(f"Sample well: {df_plate['well'].iloc[0]}, source: {df_plate['source'].iloc[0]}")

In [ ]:
# Map wells to compounds (SMILES) and build (SMILES, embedding) pairs

if downloaded:
    all_profiles = []
    for ppath in downloaded:
        df_p = pd.read_parquet(ppath)
        all_profiles.append(df_p)
    profiles_df = pd.concat(all_profiles, ignore_index=True)
    print(f"Loaded profiles: {len(profiles_df)} wells from {len(downloaded)} plates")

    emb_dim = len(np.array(profiles_df["all_emb"].iloc[0]))
    print(f"Embedding dim: {emb_dim}")

    # Map profile columns to well metadata columns
    profiles_df["Metadata_Source"] = profiles_df["source"]
    profiles_df["Metadata_Plate"] = profiles_df["plate"]
    profiles_df["Metadata_Well"] = profiles_df["well"]

    # Filter well metadata for source_4 compound wells
    source4_wells = well_df[
        (well_df["Metadata_Source"] == "source_4") &
        (well_df["Metadata_JCP2022"].str.startswith("JCP2022_", na=False))
    ].copy()
    print(f"\nSource_4 compound wells in metadata: {len(source4_wells)}")

    # Merge with compound metadata for SMILES
    compound_map = compound_df[["Metadata_JCP2022", "Metadata_InChIKey", "Metadata_SMILES"]].drop_duplicates()
    source4_with_smiles = source4_wells.merge(compound_map, on="Metadata_JCP2022", how="inner")
    print(f"Wells with SMILES: {len(source4_with_smiles)}")

    # Join profiles with SMILES
    joined = profiles_df.merge(
        source4_with_smiles[["Metadata_Source", "Metadata_Plate", "Metadata_Well",
                              "Metadata_JCP2022", "Metadata_SMILES"]],
        on=["Metadata_Source", "Metadata_Plate", "Metadata_Well"],
        how="inner"
    )
    # Drop rows with missing SMILES
    joined = joined.dropna(subset=["Metadata_SMILES"]).reset_index(drop=True)
    print(f"\nJoined rows with valid SMILES: {len(joined)}")
    print(f"Unique compounds: {joined['Metadata_JCP2022'].nunique()}")

    if len(joined) > 0:
        sample = joined.iloc[0]
        sample_emb = np.array(sample["all_emb"])
        print(f"\nSample:")
        print(f"  SMILES: {str(sample['Metadata_SMILES'])[:80]}")
        print(f"  Embedding dim: {len(sample_emb)}")
        print(f"  First 5 values: {sample_emb[:5]}")
    else:
        print("\nNo compound matches. Checking profile source/plate values:")
        print(f"  Profile sources: {profiles_df['source'].unique()}")
        print(f"  Profile plates: {profiles_df['plate'].unique()[:5]}")
        print(f"  Well plates (source_4): {source4_with_smiles['Metadata_Plate'].unique()[:5]}")
else:
    print("No downloaded plates.")

#### 2.4 Visualize CPCNN embedding statistics and compound-level profiles

In [ ]:
import matplotlib.pyplot as plt

if len(joined) > 0:
    # Extract embedding matrix from the all_emb column
    emb_matrix = np.stack(joined["all_emb"].apply(np.array).values).astype(np.float32)
    emb_dim = emb_matrix.shape[1]
    print(f"Embedding matrix: {emb_matrix.shape}")

    # Aggregate to compound level (mean over replicate wells)
    compound_groups = joined.groupby("Metadata_JCP2022")
    compound_embs = {}
    compound_smiles_map = {}
    for jcp_id, group in compound_groups:
        embs = np.stack(group["all_emb"].apply(np.array).values)
        compound_embs[jcp_id] = embs.mean(axis=0)
        compound_smiles_map[jcp_id] = group["Metadata_SMILES"].iloc[0]

    compound_emb_matrix = np.stack(list(compound_embs.values()))
    n_compounds = len(compound_embs)
    print(f"Compound-level profiles: {n_compounds} compounds x {emb_dim} dims")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Embedding dimension distribution
    ax = axes[0, 0]
    means = compound_emb_matrix.mean(axis=0)
    ax.bar(range(len(means)), means, alpha=0.7, width=1.0)
    ax.set_xlabel("Embedding dimension")
    ax.set_ylabel("Mean value (across compounds)")
    ax.set_title(f"CPCNN Embedding Dim Means ({emb_dim}d)")

    # 2. Per-compound norm distribution
    ax = axes[0, 1]
    norms = np.linalg.norm(compound_emb_matrix, axis=1)
    ax.hist(norms, bins=30, edgecolor="black", alpha=0.7)
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("Count")
    ax.set_title(f"Compound Embedding Norms (n={n_compounds})")
    ax.axvline(norms.mean(), color="red", linestyle="--", label=f"mean={norms.mean():.1f}")
    ax.legend()

    # 3. Pairwise cosine similarity
    from scipy.spatial.distance import cdist
    ax = axes[1, 0]
    n_sample = min(200, n_compounds)
    idx = np.random.RandomState(42).choice(n_compounds, n_sample, replace=False)
    sample_emb = compound_emb_matrix[idx]
    cos_sim = 1 - cdist(sample_emb, sample_emb, metric="cosine")
    off_diag = cos_sim[np.triu_indices(n_sample, k=1)]
    ax.hist(off_diag, bins=50, edgecolor="black", alpha=0.7)
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Count")
    ax.set_title(f"Inter-compound Cosine Similarity (n={n_sample})")
    ax.axvline(off_diag.mean(), color="red", linestyle="--", label=f"mean={off_diag.mean():.3f}")
    ax.legend()

    # 4. PCA of compound embeddings
    from sklearn.decomposition import PCA
    ax = axes[1, 1]
    pca = PCA(n_components=2)
    proj = pca.fit_transform(compound_emb_matrix)
    ax.scatter(proj[:, 0], proj[:, 1], alpha=0.5, s=10)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title(f"PCA of CPCNN Embeddings ({n_compounds} compounds)")

    plt.suptitle("JUMP CPCNN Embeddings (672d, U2OS Cell Painting)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # Summary stats
    print(f"\n{'='*50}")
    print(f"CPCNN Embedding Stats (pilot: {N_PILOT_PLATES} plates)")
    print(f"{'='*50}")
    print(f"  Wells total:       {len(joined)}")
    print(f"  Unique compounds:  {n_compounds}")
    print(f"  Embedding dims:    {emb_dim}")
    print(f"  Norm range:        {norms.min():.1f} — {norms.max():.1f} (mean {norms.mean():.1f})")
    print(f"  Cosine sim range:  {off_diag.min():.3f} — {off_diag.max():.3f} (mean {off_diag.mean():.3f})")
    print(f"  PCA var explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%")
else:
    print("No joined data available. Check S3 download and metadata join.")

---
### Part 3: Comparison with existing pre-training targets

Compare the new embedding sources with the existing ones in terms of coverage, dimensionality, and information content.

In [ ]:
# Summary of all pre-training targets (updated)
pretrain_targets = pd.DataFrame([
    {"Source": "L1000 (VCAP)", "Type": "Gene expression", "Dim": 2934, "Compounds": "~7K", "Cell line": "VCAP", "Status": "Active"},
    {"Source": "L1000 (MCF7)", "Type": "Gene expression", "Dim": 2934, "Compounds": "~7K", "Cell line": "MCF7", "Status": "Active"},
    {"Source": "PCBA-1328", "Type": "Bioactivity", "Dim": 1328, "Compounds": "~400K", "Cell line": "N/A (assay)", "Status": "Active"},
    {"Source": "PCQM4M", "Type": "Quantum properties", "Dim": 1, "Compounds": "~3.8M", "Cell line": "N/A (DFT)", "Status": "Active"},
    {"Source": "RxRx3", "Type": "Cell morphology", "Dim": 384, "Compounds": "~17K", "Cell line": "HUVEC", "Status": "Active"},
    {"Source": "DTI (ESM2)", "Type": "Protein binding", "Dim": 2560, "Compounds": "~100K", "Cell line": "N/A (protein)", "Status": "Active"},
    {"Source": "MACE-OFF", "Type": "3D conformational", "Dim": 448, "Compounds": "Any SMILES", "Cell line": "N/A (physics)", "Status": "New"},
    # {"Source": "JUMP CPCNN (U2OS)", "Type": "Cell morphology", "Dim": 672, "Compounds": "~116K", "Cell line": "U2OS", "Status": "New"},
])

# print(pretrain_targets.to_string(index=False))
# print("\n--- Key advantages of new targets ---")
# print("MACE-OFF:   3D physics-based (448d), universal (any SMILES), complements 2D graph learning")
# print("JUMP CPCNN: U2OS cell morphology (672d), 116K compounds — 7× more than RxRx3, different cell line")

---
### Next steps

1. **MACE-OFF**: 
   - Run batch extraction on a large compound set (PCBA SMILES or full ADMET set)
   - Create Hydra task config (`tasks/mace.yaml`) following the RxRx3 pattern
   - Test as a standalone pre-training target, then combine with existing targets

2. **BBBC047 / U2OS**:
   - Download profiles from Cell Painting Gallery or BBBC website
   - Map compound IDs to SMILES (PubChem/ChEMBL)
   - Dimensionality reduction if > 1000 features (PCA to ~384 dims to match RxRx3)
   - Create Hydra task config (`tasks/bbbc047.yaml`)

3. **Combined pre-training**:
   - `tasks=largemix_mace` — LargeMix + MACE-OFF
   - `tasks=largemix_rxrx3_bbbc047` — LargeMix + RxRx3 + BBBC047 (two cell lines)
   - `tasks=largemix_rxrx3_mace_dti` — Full multi-modal